# 0825_peace_006_type_expert_sequential_update

타입별 XGBoost 전문가 모델에 시간순 순차 업데이트를 실제 학습·추론 방식으로 적용한 실험입니다.

- 0~30% 구간에서 타입별 200개 트리를 최초 학습합니다.
- 30~40%, 40~50%, 50~60%, 60~70% 배치마다 50개 트리를 이어 붙여 최종 400개 트리로 만듭니다.
- 전처리기는 최초 0~30%에서만 fit하고 이후 모든 update와 추론에 고정합니다.
- 최종 Validation/Test는 순차 업데이트가 끝난 모델 자체로 평가합니다.
- 분할·피처·평가 지표는 `0825_peace_004_type_expert_walk_forward`와 동일합니다.


## 1. 설정, 경로 탐색과 실행 로그

In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_006_type_expert_sequential_update"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


2026-08-25 02:17:14,806 | INFO | experiment=0825_peace_006_type_expert_sequential_update


2026-08-25 02:17:14,807 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99


2026-08-25 02:17:14,812 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 02:17:14,821 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 02:17:14,822 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 02:17:14,823 | INFO | log_file=docs/peace/0825_peace_006_type_expert_sequential_update.log


log saved to: docs/peace/0825_peace_006_type_expert_sequential_update.log


## 2. 원본 데이터와 매핑 검증

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 02:17:19,135 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


## 3. 타입별 유효 피처

In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 02:17:19,144 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 4. 동일한 시간순 Train/Validation/Test 분할

In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
evaluation_policy = pd.Series(
    {
        "model_selection_uses_test": False,
        "threshold_selected_on_test": False,
        "fixed_test_threshold": DECISION_THRESHOLD,
    },
    name="evaluation_policy",
)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy model_selection=False threshold=%.2f", DECISION_THRESHOLD)


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


model_selection_uses_test     False
threshold_selected_on_test    False
fixed_test_threshold            0.5
Name: evaluation_policy, dtype: object

2026-08-25 02:17:19,502 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 02:17:19,502 | INFO | test_policy model_selection=False threshold=0.50


## 5. 동일한 평가 지표와 임계값 선택 함수

In [5]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    """Recall 제약을 만족하며 False Call Reduction이 최대인 threshold를 선택한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


# 최적화 구현이 작은 합성 예제의 완전 탐색과 같은 결과인지 검증한다.
_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")

def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def evaluate_calibration_and_future(calibration_frame, calibration_probability, evaluation_frame, evaluation_probability, stage_name):
    global_selection = select_threshold(calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL)
    type_thresholds = {}
    type_prediction = pd.Series(np.nan, index=evaluation_frame.index, dtype="float64")
    threshold_rows = [{"stage": stage_name, "scope": "global", **global_selection}]
    type_rows = []
    for inspection_type in inspection_types:
        type_calibration = calibration_frame.loc[calibration_frame[TYPE_COLUMN] == inspection_type]
        selection = select_threshold(type_calibration[TARGET], calibration_probability.loc[type_calibration.index], min_recall=MIN_RECALL)
        type_thresholds[inspection_type] = selection["threshold"]
        threshold_rows.append({"stage": stage_name, "scope": f"type_{inspection_type}", **selection})
        type_evaluation = evaluation_frame.loc[evaluation_frame[TYPE_COLUMN] == inspection_type]
        type_probability = evaluation_probability.loc[type_evaluation.index]
        prediction = (type_probability >= selection["threshold"]).astype("int8")
        type_prediction.loc[type_evaluation.index] = prediction
        metrics = evaluate_predictions(type_evaluation[TARGET], prediction, type_probability)
        type_rows.append({"stage": stage_name, "inspection_type": inspection_type, "threshold": selection["threshold"], **metrics})
    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD),
        "global_threshold": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, global_selection["threshold"]),
        "type_specific_thresholds": evaluate_predictions(evaluation_frame[TARGET], type_prediction, evaluation_probability),
    }
    metric_rows = [{"stage": stage_name, "strategy": strategy, **metrics} for strategy, metrics in strategy_metrics.items()]
    return {"global_selection": global_selection, "type_thresholds": type_thresholds, "threshold_rows": threshold_rows, "type_rows": type_rows, "metric_rows": metric_rows}

2026-08-25 02:17:19,530 | INFO | threshold_selector_unit_test=PASS


## 6. 동일한 3-Fold Expanding Walk-forward 구간

In [6]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ],
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 02:17:20,202 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

## 7. 타입별 순차 업데이트 학습

In [7]:
INITIAL_TREES = 200
UPDATE_TREES = 50
INITIAL_END = 0.30
UPDATE_WINDOWS = [(0.30, 0.40), (0.40, 0.50), (0.50, 0.60), (0.60, 0.70)]
EXPECTED_FINAL_TREES = INITIAL_TREES + UPDATE_TREES * len(UPDATE_WINDOWS)
assert EXPECTED_FINAL_TREES == XGB_PARAMS["n_estimators"]

sequential_predictions = {}
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    sequential_predictions[f"{fold_name}_calibration"] = pd.Series(np.nan, index=walk_forward_segments[fold_name]["calibration"].index, dtype="float64")
    sequential_predictions[f"{fold_name}_evaluation"] = pd.Series(np.nan, index=walk_forward_segments[fold_name]["evaluation"].index, dtype="float64")
sequential_predictions["final_validation"] = pd.Series(np.nan, index=validation_df.index, dtype="float64")
sequential_predictions["final_test"] = pd.Series(np.nan, index=test_df.index, dtype="float64")

def predict_type_into(model, preprocessor, feature_columns, frame, inspection_type, output):
    type_frame = frame.loc[frame[TYPE_COLUMN] == inspection_type]
    X_frame = preprocessor.transform(type_frame[feature_columns])
    output.loc[type_frame.index] = model.predict_proba(X_frame)[:, 1]

sequential_training_rows = []
final_models_by_type, final_preprocessors_by_type = {}, {}
for inspection_type in inspection_types:
    feature_columns = feature_columns_by_type[inspection_type]
    initial_frame = raw_df.loc[(raw_df[TIME_COLUMN] <= walk_forward_boundaries[INITIAL_END]) & (raw_df[TYPE_COLUMN] == inspection_type)]
    y_initial = initial_frame[TARGET].astype("int8")
    assert y_initial.nunique() == 2
    preprocessor = make_preprocessor(feature_columns)
    X_initial = preprocessor.fit_transform(initial_frame[feature_columns])
    model = XGBClassifier(**{**XGB_PARAMS, "n_estimators": INITIAL_TREES})
    model.fit(X_initial, y_initial, verbose=False)
    assert model.get_booster().num_boosted_rounds() == INITIAL_TREES
    sequential_training_rows.append({"stage": "initial_0_30", "inspection_type": inspection_type, "batch_rows": len(initial_frame), "batch_positive": int(y_initial.sum()), "total_trees": INITIAL_TREES, "encoded_features": X_initial.shape[1]})
    predict_type_into(model, preprocessor, feature_columns, walk_forward_segments["fold_1"]["calibration"], inspection_type, sequential_predictions["fold_1_calibration"])
    predict_type_into(model, preprocessor, feature_columns, walk_forward_segments["fold_1"]["evaluation"], inspection_type, sequential_predictions["fold_1_evaluation"])

    for update_index, (start_fraction, end_fraction) in enumerate(UPDATE_WINDOWS, start=1):
        update_frame = raw_df.loc[
            (raw_df[TIME_COLUMN] > walk_forward_boundaries[start_fraction]) &
            (raw_df[TIME_COLUMN] <= walk_forward_boundaries[end_fraction]) &
            (raw_df[TYPE_COLUMN] == inspection_type)
        ]
        y_update = update_frame[TARGET].astype("int8")
        X_update = preprocessor.transform(update_frame[feature_columns])
        updated_model = XGBClassifier(**{**XGB_PARAMS, "n_estimators": UPDATE_TREES})
        updated_model.fit(X_update, y_update, xgb_model=model.get_booster(), verbose=False)
        model = updated_model
        expected_trees = INITIAL_TREES + UPDATE_TREES * update_index
        assert model.get_booster().num_boosted_rounds() == expected_trees
        sequential_training_rows.append({"stage": f"update_{int(start_fraction*100)}_{int(end_fraction*100)}", "inspection_type": inspection_type, "batch_rows": len(update_frame), "batch_positive": int(y_update.sum()), "total_trees": expected_trees, "encoded_features": X_update.shape[1]})
        logger.info("sequential_update_done type=%d window=%.2f_%.2f rows=%d positive=%d total_trees=%d", inspection_type, start_fraction, end_fraction, len(update_frame), int(y_update.sum()), expected_trees)
        if np.isclose(end_fraction, 0.40):
            predict_type_into(model, preprocessor, feature_columns, walk_forward_segments["fold_2"]["calibration"], inspection_type, sequential_predictions["fold_2_calibration"])
            predict_type_into(model, preprocessor, feature_columns, walk_forward_segments["fold_2"]["evaluation"], inspection_type, sequential_predictions["fold_2_evaluation"])
        elif np.isclose(end_fraction, 0.50):
            predict_type_into(model, preprocessor, feature_columns, walk_forward_segments["fold_3"]["calibration"], inspection_type, sequential_predictions["fold_3_calibration"])
            predict_type_into(model, preprocessor, feature_columns, walk_forward_segments["fold_3"]["evaluation"], inspection_type, sequential_predictions["fold_3_evaluation"])
        del X_update
        gc.collect()
    assert model.get_booster().num_boosted_rounds() == EXPECTED_FINAL_TREES
    predict_type_into(model, preprocessor, feature_columns, validation_df, inspection_type, sequential_predictions["final_validation"])
    predict_type_into(model, preprocessor, feature_columns, test_df, inspection_type, sequential_predictions["final_test"])
    final_models_by_type[inspection_type] = model
    final_preprocessors_by_type[inspection_type] = preprocessor
    del X_initial
    gc.collect()
for name, probability in sequential_predictions.items():
    assert probability.notna().all(), name
sequential_training_summary = pd.DataFrame(sequential_training_rows).set_index(["stage", "inspection_type"])
display(sequential_training_summary)

2026-08-25 02:17:20,488 | INFO | sequential_update_done type=0 window=0.30_0.40 rows=8408 positive=11 total_trees=250


2026-08-25 02:17:20,571 | INFO | sequential_update_done type=0 window=0.40_0.50 rows=6496 positive=50 total_trees=300


2026-08-25 02:17:20,665 | INFO | sequential_update_done type=0 window=0.50_0.60 rows=8985 positive=14 total_trees=350


2026-08-25 02:17:20,732 | INFO | sequential_update_done type=0 window=0.60_0.70 rows=12107 positive=4 total_trees=400


2026-08-25 02:17:21,132 | INFO | sequential_update_done type=1 window=0.30_0.40 rows=3868 positive=20 total_trees=250


...

2026-08-25 02:17:23,323 | INFO | sequential_update_done type=4 window=0.30_0.40 rows=836 positive=4 total_trees=250


2026-08-25 02:17:23,365 | INFO | sequential_update_done type=4 window=0.40_0.50 rows=325 positive=2 total_trees=300


2026-08-25 02:17:23,406 | INFO | sequential_update_done type=4 window=0.50_0.60 rows=698 positive=3 total_trees=350


2026-08-25 02:17:23,439 | INFO | sequential_update_done type=4 window=0.60_0.70 rows=344 positive=0 total_trees=400


,,batch_rows,batch_positive,total_trees,encoded_features
stage,inspection_type,,,,
initial_0_30,0,28277,32,200,80
update_30_40,0,8408,11,250,80
update_40_50,0,6496,50,300,80
update_50_60,0,8985,14,350,80
update_60_70,0,12107,4,400,80
initial_0_30,1,22698,269,200,106
update_30_40,1,3868,20,250,106
update_40_50,1,2618,186,300,106
update_50_60,1,5023,80,350,106


## 8. Walk-forward 미래 Evaluation 결과

In [8]:
walk_threshold_rows, walk_metric_rows, walk_type_rows = [], [], []
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    result = evaluate_calibration_and_future(
        walk_forward_segments[fold_name]["calibration"], sequential_predictions[f"{fold_name}_calibration"],
        walk_forward_segments[fold_name]["evaluation"], sequential_predictions[f"{fold_name}_evaluation"], fold_name,
    )
    walk_threshold_rows.extend(result["threshold_rows"])
    walk_metric_rows.extend(result["metric_rows"])
    walk_type_rows.extend(result["type_rows"])

walk_forward_threshold_summary = pd.DataFrame(walk_threshold_rows).set_index(["stage", "scope"])
walk_forward_evaluation_metrics = pd.DataFrame(walk_metric_rows).set_index(["stage", "strategy"])
walk_forward_type_evaluation = pd.DataFrame(walk_type_rows).set_index(["stage", "inspection_type"])
walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index().groupby("strategy").agg(
        folds=("stage", "nunique"), mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"), min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"), total_fn=("fn", "sum"),
    )
)
display(walk_forward_threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_evaluation_metrics[["positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(walk_forward_type_evaluation[["threshold", "positive_samples", "pr_auc", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_strategy_summary)
logger.info("walk_forward_strategy_summary=%s", walk_forward_strategy_summary.to_dict(orient="index"))

threshold  positive_samples    recall  false_call_reduction  \
stage  scope                                                                 
fold_1 global   0.000170               200  0.990000              0.013819   
       type_0   0.002523                11  1.000000              0.387638   
       type_1   0.002101                20  1.000000              0.323285   
       type_2   0.000132                92  1.000000              0.003852   
       type_3   0.000931                73  1.000000              0.168828   
       type_4   0.002452                 4  1.000000              0.000000   
fold_2 global   0.000602               326  0.990798              0.186988   
       type_0   0.000436                50  1.000000              0.250698   
       type_1   0.000600               186  0.994624              0.038651   
       type_2   0.001462                49  1.000000              0.257573   
       type_3   0.038028                39  1.000000              0.629504   
       type_4   0.003616                 2  1.000000              0.000000   
fold_3 global   0.000013               152  0.993421              0.173498   
       type_0   0.001713                14  1.000000              0.676848   
       type_1   0.005249                80  1.000000              0.615618   
       type_2   0.000067                32  1.000000              0.081246   
       type_3   0.000012                23  1.000000              0.363106   
       type_4   0.004368                 3  1.000000              0.000000   

                tp  fn  
stage  scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
stage  strategy                                                          
fold_1 fixed_0.5                              326  0.121789   0.253425   
       global_threshold                       326  0.121789   0.007556   
       type_specific_thresholds               326  0.121789   0.008784   
fold_2 fixed_0.5                              152  0.010360   0.004658   
       global_threshold                       152  0.010360   0.005141   
       type_specific_thresholds               152  0.010360   0.006038   
fold_3 fixed_0.5                               39  0.053154   0.000000   
       global_threshold                        39  0.053154   0.001053   
       type_specific_thresholds                39  0.053154   0.001275   

                                   recall  false_call_reduction        f1  \
stage  strategy                                                             
fold_1 fixed_0.5                 0.113497              0.997507  0.156780   
       global_threshold          1.000000              0.020520  0.014999   
       type_specific_thresholds  0.978528              0.176534  0.017412   
fold_2 fixed_0.5                 0.019737              0.985443  0.007538   
       global_threshold          0.934211              0.375951  0.010225   
       type_specific_thresholds  0.835526              0.525264  0.011990   
fold_3 fixed_0.5                 0.000000              0.999886  0.000000   
       global_threshold          0.974359              0.177318  0.002104   
       type_specific_thresholds  0.948718              0.338522  0.002547   

                                  tp   fn     fp     tn  
stage  strategy                                          
fold_1 fixed_0.5                  37  289    109  43605  
       global_threshold          326    0  42817    897  
       type_specific_thresholds  319    7  35997   7717  
fold_2 fixed_0.5                   3  149    641  43394  
       global_threshold          142   10  27480  16555  
       type_specific_thresholds  127   25  20905  23130  
fold_3 fixed_0.5                   0   39      5  43809  
       global_threshold           38    1  36045   7769  
       type_specific_thresholds   37    2  28982  14832

threshold  positive_samples    pr_auc    recall  \
stage  inspection_type                                                    
fold_1 0                 0.002523                50  0.024155  0.940000   
       1                 0.002101               186  0.230911  0.978495   
       2                 0.000132                49  0.296447  1.000000   
       3                 0.000931                39  0.496629  1.000000   
       4                 0.002452                 2  0.006154  1.000000   
fold_2 0                 0.000436                14  0.006058  1.000000   
       1                 0.000600                80  0.275353  1.000000   
       2                 0.001462                32  0.024217  0.875000   
       3                 0.038028                23  0.001217  0.086957   
       4                 0.003616                 3  0.004298  1.000000   
fold_3 0                 0.001713                 4  0.002842  1.000000   
       1                 0.005249                25  0.060241  1.000000   
       2                 0.000067                 7  0.279123  0.857143   
       3                 0.000012                 3  0.051284  0.666667   
       4                 0.004368                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
stage  inspection_type                                 
fold_1 0                            0.703847   47   3  
       1                            0.226151  182   4  
       2                            0.005445   49   0  
       3                            0.162008   39   0  
       4                            0.000000    2   0  
fold_2 0                            0.511872   14   0  
       1                            0.307910   80   0  
       2                            0.225350   28   4  
       3                            0.726452    2  21  
       4                            0.000000    3   0  
fold_3 0                            0.389738    4   0  
       1                            0.304413   25   0  
       2                            0.078623    6   1  
       3                            0.599132    2   1  
       4                            0.000000    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.061768,0.044411,0.000000,0,0.994279,0.985443,40,477
global_threshold,3,0.061768,0.969523,0.934211,1,0.191263,0.020520,506,11
type_specific_thresholds,3,0.061768,0.920924,0.835526,0,0.346773,0.176534,483,34


2026-08-25 02:17:24,241 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.06176762698621216, 'mean_recall': 0.04441125820686686, 'min_recall': 0.0, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9942785993910795, 'min_false_call_reduction': 0.9854433972976042, 'total_tp': 40, 'total_fn': 477}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.06176762698621216, 'mean_recall': 0.9695231668915879, 'min_recall': 0.9342105263157895, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.1912628141283891, 'min_false_call_reduction': 0.02051974195909777, 'total_tp': 506, 'total_fn': 11}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.06176762698621216, 'mean_recall': 0.9209239572897951, 'min_recall': 0.8355263157894737, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.34677325391092606, 'min_false_call_reduction': 0.17653383355446767, 'total_tp': 483, 'total_fn': 34}}


## 9. 최종 Validation 임계값 선택과 Walk-forward 모델의 Test 추론

In [9]:
validation_probability = sequential_predictions["final_validation"]
test_probability = sequential_predictions["final_test"]
model_summary = pd.Series({"initial_trees": INITIAL_TREES, "trees_per_update": UPDATE_TREES, "updates": len(UPDATE_WINDOWS), "final_trees_per_type": EXPECTED_FINAL_TREES}, name="final_sequential_model")

final_result = evaluate_calibration_and_future(validation_df, validation_probability, test_df, test_probability, "final_test")
global_threshold_selection = final_result["global_selection"]
thresholds_by_type = final_result["type_thresholds"]
threshold_summary = pd.DataFrame(final_result["threshold_rows"]).set_index(["stage", "scope"])
type_selected_test_metrics = pd.DataFrame(final_result["type_rows"]).set_index(["stage", "inspection_type"])
test_strategy_metrics = pd.DataFrame(final_result["metric_rows"]).set_index(["stage", "strategy"])
validation_fixed_metrics = pd.Series(evaluate_probabilities(validation_df[TARGET], validation_probability), name="validation_fixed_0.5")
type_validation_rows, type_test_rows = [], []
for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    type_validation_rows.append({"inspection_type": inspection_type, **evaluate_probabilities(type_validation[TARGET], validation_probability.loc[type_validation.index])})
    type_test_rows.append({"inspection_type": inspection_type, **evaluate_probabilities(type_test[TARGET], test_probability.loc[type_test.index])})
type_validation_metrics = pd.DataFrame(type_validation_rows).set_index("inspection_type")
type_test_metrics = pd.DataFrame(type_test_rows).set_index("inspection_type")
display(model_summary)
display(threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(test_strategy_metrics[["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(type_selected_test_metrics[["threshold", "positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(type_validation_metrics)
display(type_test_metrics)
logger.info("validation_fixed_metrics=%s", validation_fixed_metrics.to_dict())
logger.info("test_strategy_metrics=%s", test_strategy_metrics.reset_index().to_dict(orient="records"))

initial_trees           200
trees_per_update         50
updates                   4
final_trees_per_type    400
Name: final_sequential_model, dtype: int64

threshold  positive_samples    recall  \
stage      scope                                           
final_test global   0.000071               357  0.991597   
           type_0   0.000067                12  1.000000   
           type_1   0.001634               224  0.991071   
           type_2   0.000025                27  1.000000   
           type_3   0.000069                21  1.000000   
           type_4   0.002940                73  1.000000   

                   false_call_reduction   tp  fn     fp     tn  
stage      scope                                                
final_test global              0.426275  354   3  25054  18615  
           type_0              0.096332   12   0  11998   1279  
           type_1              0.601000  222   2   2473   3725  
           type_2              0.356462   27   0   4591   2543  
           type_3              0.824225   21   0   2853  13378  
           type_4              0.000000   73   0    829      0

pr_auc  precision    recall  \
stage      strategy                                                  
final_test fixed_0.5                 0.277874   0.504026  0.134624   
           global_threshold          0.277874   0.041287  0.944516   
           type_specific_thresholds  0.277874   0.042569  0.941935   

                                     false_call_reduction        f1    tp  \
stage      strategy                                                         
final_test fixed_0.5                             0.996407  0.212492   313   
           global_threshold                      0.405170  0.079115  2196   
           type_specific_thresholds              0.425432  0.081457  2190   

                                       fn     fp     tn  
stage      strategy                                      
final_test fixed_0.5                 2012    308  85419  
           global_threshold           129  50993  34734  
           type_specific_thresholds   135  49256  36471

threshold  positive_samples    pr_auc  precision  \
stage      inspection_type                                                     
final_test 0                 0.000067               195  0.030243   0.012636   
           1                 0.001634               774  0.492030   0.111392   
           2                 0.000025               731  0.566885   0.038683   
           3                 0.000069               612  0.138433   0.049739   
           4                 0.002940                13  0.017857   0.017857   

                              recall  false_call_reduction   tp  fn     fp  \
stage      inspection_type                                                   
final_test 0                0.974359              0.230618  190   5  14846   
           1                0.966408              0.484581  748  26   5967   
           2                0.965800              0.114426  706  25  17545   
           3                0.870915              0.703353  533  79  10183   
           4                1.000000              0.000000   13   0    715   

                               tn  
stage      inspection_type         
final_test 0                 4450  
           1                 5610  
           2                 2267  
           3                24144  
           4                    0

,rows,positive_samples,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
inspection_type,,,,,,,,,,,,,
0,13289,12,13277,0,12,0,0.999097,0.000000,0.000000,1.000000,0.000000,0.759587,0.003588
1,6422,224,6117,81,120,104,0.968701,0.562162,0.464286,0.986931,0.508557,0.952650,0.553935
2,7161,27,7134,0,27,0,0.996230,0.000000,0.000000,1.000000,0.000000,0.859813,0.292221
3,16252,21,16231,0,21,0,0.998708,0.000000,0.000000,1.000000,0.000000,0.961954,0.249621
4,902,73,829,0,73,0,0.919069,0.000000,0.000000,1.000000,0.000000,0.500000,0.080931


,rows,positive_samples,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
inspection_type,,,,,,,,,,,,,
0,19491,195,19296,0,195,0,0.989995,0.000000,0.000000,1.000000,0.000000,0.724331,0.030243
1,12351,774,11269,308,462,312,0.937657,0.503226,0.403101,0.973396,0.447633,0.868582,0.492030
2,20543,731,19812,0,730,1,0.964465,1.000000,0.001368,1.000000,0.002732,0.888851,0.566885
3,34939,612,34327,0,612,0,0.982484,0.000000,0.000000,1.000000,0.000000,0.861354,0.138433
4,728,13,715,0,13,0,0.982143,0.000000,0.000000,1.000000,0.000000,0.500000,0.017857


2026-08-25 02:17:24,847 | INFO | validation_fixed_metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43588.0, 'fp': 81.0, 'fn': 253.0, 'tp': 104.0, 'accuracy': 0.9924135737973016, 'precision': 0.5621621621621622, 'recall': 0.2913165266106443, 'false_call_reduction': 0.9981451372827406, 'f1': 0.3837638376383764, 'roc_auc': 0.9467575759150211, 'pr_auc': 0.3852052142795444}


2026-08-25 02:17:24,848 | INFO | test_strategy_metrics=[{'stage': 'final_test', 'strategy': 'fixed_0.5', 'rows': 88052, 'positive_samples': 2325, 'tn': 85419, 'fp': 308, 'fn': 2012, 'tp': 313, 'accuracy': 0.9736519329487121, 'precision': 0.5040257648953301, 'recall': 0.13462365591397848, 'false_call_reduction': 0.9964071995987261, 'f1': 0.21249151391717583, 'roc_auc': 0.8643120553605338, 'pr_auc': 0.27787376275195047}, {'stage': 'final_test', 'strategy': 'global_threshold', 'rows': 88052, 'positive_samples': 2325, 'tn': 34734, 'fp': 50993, 'fn': 129, 'tp': 2196, 'accuracy': 0.41941125698450915, 'precision': 0.041286732219067854, 'recall': 0.944516129032258, 'false_call_reduction': 0.4051698997981966, 'f1': 0.07911517815325864, 'roc_auc': 0.8643120553605338, 'pr_auc': 0.27787376275195047}, {'stage': 'final_test', 'strategy': 'type_specific_thresholds', 'rows': 88052, 'positive_samples': 2325, 'tn': 36471, 'fp': 49256, 'fn': 135, 'tp': 2190, 'accuracy': 0.4390700949438968, 'precision': 0

## 10. 원본 무결성과 종료 확인

In [10]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE
verification = pd.Series({
    "dataset_sha256_unchanged": True, "mapping_sha256_unchanged": True,
    "trained_model_units": len(final_models_by_type), "final_trees_per_type": EXPECTED_FINAL_TREES,
    "test_evaluated_with_walk_forward_model": True,
    "fixed_threshold": DECISION_THRESHOLD,
    "global_threshold": global_threshold_selection["threshold"],
    "type_thresholds": thresholds_by_type,
    "log_file": f"docs/peace/{LOG_PATH.name}",
}, name="verification")
display(verification)
logger.info("source_integrity=PASS walk_forward_test_model=True")
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()

dataset_sha256_unchanged                                                               True
mapping_sha256_unchanged                                                               True
trained_model_units                                                                       5
final_trees_per_type                                                                    400
test_evaluated_with_walk_forward_model                                                 True
fixed_threshold                                                                         0.5
global_threshold                                                                   0.000071
type_thresholds                           {0: 6.689183646813035e-05, 1: 0.00163356412667...
log_file                                  docs/peace/0825_peace_006_type_expert_sequenti...
Name: verification, dtype: object

2026-08-25 02:17:25,009 | INFO | source_integrity=PASS walk_forward_test_model=True


2026-08-25 02:17:25,009 | INFO | experiment_complete=0825_peace_006_type_expert_sequential_update


## 11. 결론과 해석

- 최종 Test PR-AUC는 **0.277874**로, 0~30% 최초 모델에 네 시간 배치를 이어 학습한 최종 모델에서 계산됐습니다.
- Validation에서 선택한 공통 임계값은 Test Recall **94.45%**, False Call Reduction **40.52%**였습니다.
- 타입별 임계값은 Test Recall **94.19%**, False Call Reduction **42.54%**였습니다.
- Walk-forward 공통 임계값은 평균 Recall **96.95%**, 최저 Fold Recall **93.42%**로 Fold 앙상블보다 시간 안정성이 낮았습니다.
- 순차 업데이트는 타입당 최종 400개 트리만 유지해 추론 비용은 작지만, 이번 설정에서는 Fold 앙상블보다 Test PR-AUC가 낮았습니다.
